# Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"

In [3]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [4]:
# The .env file lives here, in project/starter (copy .env.example from the repo root).
# It holds OPENAI_API_KEY (the course voc- key), OPENAI_BASE_URL and TAVILY_API_KEY.
# Print only WHICH variables are set and where they point, never the keys themselves.
print("OPENAI_API_KEY :", "set" if os.getenv("OPENAI_API_KEY") else "MISSING")
print("OPENAI_BASE_URL:", os.getenv("OPENAI_BASE_URL"))
print("TAVILY_API_KEY :", "set" if os.getenv("TAVILY_API_KEY") else "MISSING")

OPENAI_API_KEY : set
OPENAI_BASE_URL: https://openai.vocareum.com/v1
TAVILY_API_KEY : set


### VectorDB Instance

In [5]:
# A persistent client: the database is written to ./chromadb (inside project/starter),
# so Part 2 can load it later. Keep this path: Part 2 loads the database from "chromadb".
chroma_client = chromadb.PersistentClient(path="chromadb")
print("ChromaDB folder: chromadb/ (inside project/starter)")

ChromaDB folder: chromadb/ (inside project/starter)


### Collection

In [6]:
# The embedding function turns each game's text into a vector (a list of numbers placed so that
# texts with similar meaning sit close together). It calls the course's OpenAI endpoint with the
# key from .env. Part 2 rebuilds exactly this function when it loads the collection.
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key_env_var="OPENAI_API_KEY",
    api_base=os.getenv("OPENAI_BASE_URL"),
)

In [7]:
class GameStoreManager:
    """A reusable manager for the game collection in ChromaDB.

    It owns the three things every later step needs: creating (or rebuilding) the collection with
    the right embedding function, opening an existing one, and searching it. Part 2 opens the same
    folder and collection name, so a store this manager builds is the one the agent uses.
    """

    def __init__(self, client, name: str, embedding_function):
        self.client = client
        self.name = name
        self.embedding_function = embedding_function
        self.collection = None

    def rebuild(self):
        """Start from an empty collection, so a re-run never sends the same documents to be embedded twice."""
        if self.name in [getattr(c, "name", c) for c in self.client.list_collections()]:
            self.client.delete_collection(self.name)
        self.collection = self.client.get_or_create_collection(name=self.name, embedding_function=self.embedding_function)
        return self.collection

    def open(self):
        """Open the collection as it is on disk (what Part 2 does)."""
        self.collection = self.client.get_collection(self.name, embedding_function=self.embedding_function)
        return self.collection

    def count(self) -> int:
        return self.collection.count()

    def search(self, query: str, n_results: int = 3) -> list:
        """The n games nearest in meaning to `query`, with their distances (smaller is closer)."""
        r = self.collection.query(query_texts=[query], n_results=n_results, include=["metadatas", "distances"])
        return [{"id": i, **m, "distance": round(d, 4)}
                for i, m, d in zip(r["ids"][0], r["metadatas"][0], r["distances"][0])]


# Keep this name: Part 2 loads the collection "udaplay".
manager = GameStoreManager(chroma_client, name="udaplay", embedding_function=embedding_fn)
collection = manager.rebuild()
print("Collection:", collection.name, "| documents before loading:", manager.count())

Collection: udaplay | documents before loading: 0


### Add documents

In [8]:
# Run this notebook from project/starter: the game files are in ./games
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # The text that gets embedded. Genre and Publisher are included as well as the starter's
    # platform, name, year and description, so questions about publishers or genres can
    # match a game by meaning, not only questions about its title.
    content = (
        f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - "
        f"Genre: {game['Genre']}. Publisher: {game['Publisher']}. {game['Description']}"
    )

    # Use file name (like 001) as ID, so an answer can cite the exact file it came from
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

print("Documents in the collection:", manager.count())

Documents in the collection: 15


### Check what was stored
The collection should hold one document per JSON file in `games/` (15). Each document keeps the
original fields as metadata, so the agent can quote them exactly.

In [9]:
json_files = sorted(f for f in os.listdir(data_dir) if f.endswith(".json"))
stored = collection.get(include=["documents", "metadatas"])
assert manager.count() == len(json_files), f"expected {len(json_files)} documents, found {manager.count()}"
print(f"{manager.count()} documents for {len(json_files)} JSON files\n")
for doc_id, meta in sorted(zip(stored["ids"], stored["metadatas"])):
    print(f"{doc_id}  {meta['Name']}  ({meta['Platform']}, {meta['YearOfRelease']})  publisher: {meta['Publisher']}")
print("\nExample of the text that was embedded (document 009):")
print(collection.get(ids=["009"], include=["documents"])["documents"][0])

15 documents for 15 JSON files

001  Gran Turismo  (PlayStation 1, 1997)  publisher: Sony Computer Entertainment
002  Grand Theft Auto: San Andreas  (PlayStation 2, 2004)  publisher: Rockstar Games
003  Gran Turismo 5  (PlayStation 3, 2010)  publisher: Sony Computer Entertainment
004  Marvel's Spider-Man  (PlayStation 4, 2018)  publisher: Sony Interactive Entertainment
005  Marvel's Spider-Man 2  (PlayStation 5, 2023)  publisher: Sony Interactive Entertainment
006  Pokémon Gold and Silver  (Game Boy Color, 1999)  publisher: Nintendo
007  Pokémon Ruby and Sapphire  (Game Boy Advance, 2002)  publisher: Nintendo
008  Super Mario World  (Super Nintendo Entertainment System (SNES), 1990)  publisher: Nintendo
009  Super Mario 64  (Nintendo 64, 1996)  publisher: Nintendo
010  Super Smash Bros. Melee  (GameCube, 2001)  publisher: Nintendo
011  Wii Sports  (Wii, 2006)  publisher: Nintendo
012  Mario Kart 8 Deluxe  (Nintendo Switch, 2017)  publisher: Nintendo
013  Kinect Adventures!  (Xbox 360, 

### Semantic search
A query is embedded with the same function and compared with every stored document. The nearest
documents come back first, with a distance (smaller means closer in meaning). None of these queries
uses a game's exact title.

In [10]:
def semantic_search(query: str, n_results: int = 3) -> None:
    """Print the n nearest games to `query`, using the manager's search."""
    print(f"Query: {query}")
    for hit in manager.search(query, n_results):
        print(f"   {hit['id']}  {hit['Name']} ({hit['Platform']}, {hit['YearOfRelease']})  distance={hit['distance']:.3f}")
    print()

semantic_search("a realistic car racing simulator")
semantic_search("the first Mario platformer in 3D")
semantic_search("pocket monsters on a handheld console")
semantic_search("open-world crime game by Rockstar")

Query: a realistic car racing simulator


   001  Gran Turismo (PlayStation 1, 1997)  distance=0.134
   003  Gran Turismo 5 (PlayStation 3, 2010)  distance=0.141
   012  Mario Kart 8 Deluxe (Nintendo Switch, 2017)  distance=0.187

Query: the first Mario platformer in 3D


   009  Super Mario 64 (Nintendo 64, 1996)  distance=0.124
   008  Super Mario World (Super Nintendo Entertainment System (SNES), 1990)  distance=0.149
   012  Mario Kart 8 Deluxe (Nintendo Switch, 2017)  distance=0.200

Query: pocket monsters on a handheld console


   006  Pokémon Gold and Silver (Game Boy Color, 1999)  distance=0.162
   007  Pokémon Ruby and Sapphire (Game Boy Advance, 2002)  distance=0.171
   009  Super Mario 64 (Nintendo 64, 1996)  distance=0.218

Query: open-world crime game by Rockstar


   002  Grand Theft Auto: San Andreas (PlayStation 2, 2004)  distance=0.152
   004  Marvel's Spider-Man (PlayStation 4, 2018)  distance=0.190
   014  Minecraft (Xbox One, 2014)  distance=0.203

